# Session 2B: Local explanations with SHAP and LIME

This notebook continues Session 2 by focusing on local explanation methods for individual predictions.

It assumes that Session 2 Codebook A introduced:
- the Adult dataset
- the fitted Random Forest pipeline
- global explanation methods such as PFI and PDP/ICE

Here we focus on:
- SHAP
- LIME
- comparing local explanations for selected test cases

Reference - 

> LIME: Ribeiro, Marco Tulio, Sameer Singh, and Carlos Guestrin. 2016. “"Why Should I Trust You?": Explaining the Predictions of Any Classifier.” In Proceedings of the 22nd ACM SIGKDD International Conference on Knowledge Discovery and Data Mining, 1135–44. KDD ’16. New York, NY, USA: Association for Computing Machinery. https://doi.org/10.1145/2939672.2939778.

> SHAP: Lundberg, Scott M., and Su-In Lee. 2017. “A Unified Approach to Interpreting Model Predictions.” In Proceedings of the 31st International Conference on Neural Information Processing Systems, 4768–77. NIPS’17. Red Hook, NY, USA: Curran Associates Inc.

Read Molnar's text to get a brief overview of how this approach evolved from Shapley values

In [ ]:
!pip install -q shap lime matplotlib

In [47]:
from pathlib import Path
import pandas as pd
import joblib
import shap
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score
import matplotlib.pyplot as plt
from lime.lime_tabular import LimeTabularExplainer
from IPython.display import display, HTML



In [2]:
def find_repo_root(start_path=None):
    if start_path is None:
        start_path = Path.cwd()

    current = start_path.resolve()
    for path in [current] + list(current.parents):
        if (path / "Data").exists() and (path / "Outputs").exists() and (path / "Code").exists():
            return path

    raise FileNotFoundError(
        "Could not find the repository root. "
        "Make sure the notebook is running inside the Explainable_Machine_Learning_XAI repo."
    )

REPO_ROOT = find_repo_root()
DATA_DIR = REPO_ROOT / "Data"
OUTPUT_DIR = REPO_ROOT / "Outputs" / "session_2"
FIG_DIR = OUTPUT_DIR / "figures"
TABLE_DIR = OUTPUT_DIR / "tables"

In [ ]:
# Reload the saved objects
session2_01_cleaned_adult_data_path = TABLE_DIR / "session2_01_cleaned_adult_data.csv"
session2_02_X_adult_train_path = TABLE_DIR / "session2_02_X_adult_train.csv"
session2_03_X_adult_test_path = TABLE_DIR / "session2_03_X_adult_test.csv"
session2_04_y_adult_train_path = TABLE_DIR / "session2_04_y_adult_train.csv"
session2_05_y_adult_test_path = TABLE_DIR / "session2_05_y_adult_test.csv"

df_adult_model = pd.read_csv(session2_01_cleaned_adult_data_path)
X_adult_train = pd.read_csv(session2_02_X_adult_train_path)
X_adult_test = pd.read_csv(session2_03_X_adult_test_path)
y_adult_train = pd.read_csv(session2_04_y_adult_train_path).squeeze("columns")
y_adult_test = pd.read_csv(session2_05_y_adult_test_path).squeeze("columns")

print("Reloaded shapes:")
print("df_adult_model:", df_adult_model.shape)
print("X_adult_train:", X_adult_train.shape)
print("X_adult_test :", X_adult_test.shape)
print("y_adult_train:", y_adult_train.shape)
print("y_adult_test :", y_adult_test.shape)

display(X_adult_test.head())
display(y_adult_test.head())

In [ ]:
# Loading the fitted Random Forest pipeline saved in Session 2A
session2_20_rf_pipeline_adult_path = OUTPUT_DIR / "session2_20_rf_pipeline_adult.joblib"
rf_pipeline_adult = joblib.load(session2_20_rf_pipeline_adult_path)

In [ ]:
# same random seed as Session 2A for consistency
SEED = 42

print("SEED:", SEED)

Quick check

In [ ]:
y_adult_test_pred_rf = rf_pipeline_adult.predict(X_adult_test)
y_adult_test_proba_rf = rf_pipeline_adult.predict_proba(X_adult_test)[:, 1]

print("Test accuracy :", round(accuracy_score(y_adult_test, y_adult_test_pred_rf), 6))
print("Test ROC AUC  :", round(roc_auc_score(y_adult_test, y_adult_test_proba_rf), 6))

# selecting cases since we will apply local method

In [ ]:
adult_test_predictions_df = X_adult_test.copy()
adult_test_predictions_df["true_income_label"] = y_adult_test.values
adult_test_predictions_df["predicted_probability_income_gt_50K"] = y_adult_test_proba_rf
adult_test_predictions_df["predicted_class"] = y_adult_test_pred_rf

adult_example_high_idx = adult_test_predictions_df["predicted_probability_income_gt_50K"].idxmax()
adult_example_low_idx = adult_test_predictions_df["predicted_probability_income_gt_50K"].idxmin()

print("High-probability example:", adult_example_high_idx)
print("Low-probability example:", adult_example_low_idx)

display(adult_test_predictions_df.loc[[adult_example_high_idx, adult_example_low_idx]])

In [11]:
# Store the two example rows for later SHAP and LIME sections
X_adult_example_high = X_adult_test.loc[[adult_example_high_idx]].copy()
X_adult_example_low = X_adult_test.loc[[adult_example_low_idx]].copy()

# SHAP

In [ ]:
# Use a small background sample from the training data
# SHAP uses this background data as a reference point for explanations.
# We sample from the training set, not the test set, because the training data
# represent the data distribution the model learned from.
X_adult_train_background_shap = X_adult_train.sample(
    n=200,
    random_state=SEED
).copy()

print("Background sample")
print(X_adult_train_background_shap.shape)

we transform:

- the SHAP background sample
- the high-probability case
- the low-probability case

into the same numeric feature space that the Random Forest used.

## Tidying up for input

In [20]:
# Extract the fitted preprocessing step and fitted Random Forest model
adult_fitted_preprocessor = rf_pipeline_adult.named_steps["preprocessor"]
adult_fitted_rf_model = rf_pipeline_adult.named_steps["model"]

# Transform the SHAP background sample into the numeric feature space
X_adult_train_background_shap_transformed = adult_fitted_preprocessor.transform(
    X_adult_train_background_shap
)

# Transform the two selected example rows into the same numeric feature space
X_adult_example_high_transformed = adult_fitted_preprocessor.transform(X_adult_example_high)
X_adult_example_low_transformed = adult_fitted_preprocessor.transform(X_adult_example_low)

# Convert to dense arrays if the transformed output is sparse
if hasattr(X_adult_train_background_shap_transformed, "toarray"):
    X_adult_train_background_shap_transformed = X_adult_train_background_shap_transformed.toarray()

if hasattr(X_adult_example_high_transformed, "toarray"):
    X_adult_example_high_transformed = X_adult_example_high_transformed.toarray()

if hasattr(X_adult_example_low_transformed, "toarray"):
    X_adult_example_low_transformed = X_adult_example_low_transformed.toarray()

# Recover the transformed feature names after preprocessing
adult_transformed_feature_names = adult_fitted_preprocessor.get_feature_names_out()

We have created two **SHAP explanation objects**:

- one for the high-probability case
- one for the low-probability case

These objects contain the feature contribution values that will be used in the later step.

In [23]:
#### Build the tree-based SHAP explainer

# Now that the background sample and the two selected cases have been transformed into the same numeric feature space used by the fitted Random Forest, we can use **TreeExplainer**.

# This is more appropriate here than the earlier generic SHAP route because we are now explaining the fitted **tree model itself** in its actual input space.

# Build a tree-based SHAP explainer for the fitted Random Forest model.
#
# Input 1:
#   adult_fitted_rf_model
#   -> the fitted Random Forest extracted from the pipeline
#
# Input 2:
#   X_adult_train_background_shap_transformed
#   -> the transformed background dataset used as the SHAP reference distribution
#
# model_output="probability":
#   -> asks SHAP to explain predicted probabilities rather than raw tree output
#
# Output:
#   adult_tree_explainer_shap
#   -> a tree-specific SHAP explainer object
adult_tree_explainer_shap = shap.TreeExplainer(
    model=adult_fitted_rf_model,
    data=X_adult_train_background_shap_transformed,
    model_output="probability"
)

In [ ]:
# Apply the tree-based SHAP explainer to the transformed high- and low-probability cases.
#
# Inputs:
#   X_adult_example_high_transformed
#   X_adult_example_low_transformed
#   -> these are the two selected rows after preprocessing
#
# What this does:
#   -> computes SHAP contribution values for each transformed feature
#   -> stores the results as arrays for each class output
#
# Outputs:
#   shap_values_example_high
#   shap_values_example_low
#
# For a binary classifier with model_output="probability", SHAP returns values
# for both classes. We will inspect their structure first before plotting.

shap_values_example_high = adult_tree_explainer_shap.shap_values(
    X_adult_example_high_transformed
)

shap_values_example_low = adult_tree_explainer_shap.shap_values(
    X_adult_example_low_transformed
)

print("\nHigh-case SHAP values shape(s):")
if isinstance(shap_values_example_high, list):
    for i, arr in enumerate(shap_values_example_high):
        print(f"Class {i}: {arr.shape}")
else:
    print(shap_values_example_high.shape)

print("\nLow-case SHAP values shape(s):")
if isinstance(shap_values_example_low, list):
    for i, arr in enumerate(shap_values_example_low):
        print(f"Class {i}: {arr.shape}")
else:
    print(shap_values_example_low.shape)

- `1` = **one explained observation**
- `108` = **108 transformed features after preprocessing** - the one hot encoding
- `2` = **two model outputs**, which in our binary classification problem are the two income classes

So SHAP has calculated contribution values for:

- one selected case
- across all transformed model features
- for both class probabilities

The interpretation comes in during the **next steps**, when we inspect:

- the explainer's `expected_value`
- the model's predicted probability for the selected row
- the SHAP values for one chosen class
- the waterfall plot

In [ ]:
# Inspect the SHAP baseline and compare it with the model prediction.
#
# expected_value:
#   the SHAP baseline output before feature contributions are added
#
# For class index 1:
#   this corresponds to the positive class: income > 50K

print("SHAP expected values:")
print(adult_tree_explainer_shap.expected_value)

# Model-predicted probabilities for the two selected cases
adult_high_case_pred_proba = adult_fitted_rf_model.predict_proba(X_adult_example_high_transformed)[0]
adult_low_case_pred_proba = adult_fitted_rf_model.predict_proba(X_adult_example_low_transformed)[0]

print("\nHigh-case predicted probabilities [<=50K, >50K]:")
print(adult_high_case_pred_proba)

print("\nLow-case predicted probabilities [<=50K, >50K]:")
print(adult_low_case_pred_proba)

# Check the SHAP additivity idea for class 1
print("\nHigh-case: baseline + SHAP sum for class 1")
print(adult_tree_explainer_shap.expected_value[1] + shap_values_example_high[0, :, 1].sum())

print("High-case: model predicted probability for class 1")
print(adult_high_case_pred_proba[1])

print("\nLow-case: baseline + SHAP sum for class 1")
print(adult_tree_explainer_shap.expected_value[1] + shap_values_example_low[0, :, 1].sum())

print("Low-case: model predicted probability for class 1")
print(adult_low_case_pred_proba[1])

## Interpreting the SHAP baseline and prediction check

SHAP explains a prediction by starting from a **baseline** called the `expected_value`, and then adding the feature contribution values until it reaches the model's final output. For multiple-output models, SHAP states:

$$
\text{shap\_values}[i, :, j].sum() + \text{expected\_value}[j] = f(x)[i, j]
$$

- $i$ = row index
- $j$ = output index
- $\text{expected\_value}[j]$ = baseline for output $j$
- $f(x)[i, j]$ = model output for row $i$ and output $j$  
  :contentReference[oaicite:0]{index=0}

### What does `SHAP expected values: [0.7399 0.2601]` mean?

These are the **baseline class probabilities** under the SHAP background data:

- `0.7399` for class `0` = `<=50K`
- `0.2601` for class `1` = `>50K`  

So before looking at any one selected person, SHAP starts from the idea that the model's baseline probability for:

$$
P(\text{income} > 50K)
$$

is about:

$$
0.2601
$$

For the high-probability case, the model predicted:

- `0.0` for `<=50K`
- `1.0` for `>50K`

So this particular person was predicted to have essentially a **100% probability of `income > 50K`** according to the fitted model.

Then SHAP checked:

$$
0.2601 + \sum \text{feature contributions for class 1} \approx 1.0
$$

and got:

$$
0.9999999910982369
$$

which is effectively the same as `1.0`, up to tiny floating-point rounding.  
This means the SHAP contributions for class `1` successfully explain how the model moved from the baseline probability `0.2601` up to the final predicted probability `1.0`. 
### What do the low-case probabilities mean?

For the low-probability case, the model predicted:

- `1.0` for `<=50K`
- `0.0` for `>50K`

So this person was predicted to have essentially a **0% probability of `income > 50K`**.

Then SHAP checked:

$$
0.2601 + \sum \text{feature contributions for class 1} \approx 0.0
$$

and got:

$$
-1.0826865515234374 \times 10^{-9}
$$

which is effectively `0.0`, again up to tiny floating-point rounding.

So the SHAP contributions for class `1` successfully explain how the model moved from the baseline probability `0.2601` down to the final predicted probability `0.0`.

For class `1` (`income > 50K`):

- the baseline probability starts at about **26%**
- for the **high case**, the feature contributions push that probability **up to 100%**
- for the **low case**, the feature contributions push that probability **down to 0%**

SHAP has explained both selected cases numerically.


# High class case plotted (waterfall plot)

In [ ]:
adult_shap_explanation_high_class1 = shap.Explanation(
    values=shap_values_example_high[0, :, 1],
    base_values=adult_tree_explainer_shap.expected_value[1],
    data=X_adult_example_high_transformed[0],
    feature_names=adult_transformed_feature_names
)

shap.plots.waterfall(adult_shap_explanation_high_class1, max_display=15)

plt.gcf().savefig(
    FIG_DIR / "session2_21_shap_waterfall_high_case_class1.png",
    dpi=300,
    bbox_inches="tight"
)

In [ ]:
adult_shap_explanation_low_class1 = shap.Explanation(
    values=shap_values_example_low[0, :, 1],
    base_values=adult_tree_explainer_shap.expected_value[1],
    data=X_adult_example_low_transformed[0],
    feature_names=adult_transformed_feature_names
)

shap.plots.waterfall(adult_shap_explanation_low_class1, max_display=15)

plt.gcf().savefig(
    FIG_DIR / "session2_02_shap_waterfall_low_case_class1.png",
    dpi=300,
    bbox_inches="tight"
)

Understanding waterfall plots -> https://shap.readthedocs.io/en/latest/example_notebooks/api_examples/plots/waterfall.html


> The feature name after the equals sign is the transformed feature name used by the model after preprocessing.

99999 means this person’s transformed numeric value for capital_gain is 99999
13 means this person’s education_num is 13
1 means that one-hot encoded category is present
0 means that one-hot encoded category is absent


> Because categorical variables were one-hot encoded, the plot now shows transformed names such as:

cat__education_Bachelors
cat__relationship_Husband
cat__marital_status_Never-married

> The bars

- pink/red bars = positive contributions
- blue bars = negative contributions

So in the high-case plot, the bars are mostly pink because the features are pushing the probability of >50K upward.

In the low-case plot, the bars are mostly blue because the features are pushing the probability of >50K downward.

//bars pointing the prediction to the right / upward in value mean stronger support for >50K//

//bars pulling the prediction left / downward in value mean weaker support for >50K//

Values inside the bars like these

+0.40 -> example +0.40 means that feature increased the class probability by about 0.40

+0.06

-0.06 -> -0.06 means that feature decreased the class probability by about 0.06

-0.03

are the SHAP values for those individual features. show how much that feature moved the prediction on the output scale being explained, starting at 0.26 with shifts.

###  SHAP waterfall plot, the whole idea

We have numerical capture earlier but plotting the SHAP against the features like this for some select cases really breaks down the items

A SHAP waterfall plot explains **one prediction for one class**.

In our notebook, that class is:

$$
\text{income} > 50K
$$

The plot shows how the model moves from a **baseline prediction** to the **final prediction** for one selected person. SHAP’s documentation describes waterfall plots as starting at the expected value of the model output, then adding positive and negative feature contributions until the model output for that prediction is reached. :contentReference

---

#### 1. The baseline on the x-axis: $E[f(X)] = 0.26$

At the bottom of the plot you see something like:

$$
E[f(X)] = 0.26
$$

This is the **expected value** for the chosen class. It is the SHAP baseline, meaning the reference prediction before the model looks at this specific person. For our class `>50K`, that baseline is about:

$$
0.2601
$$

So the plot always starts from this reference level.

---

#### 2. The final value at the top: $f(x)$

At the top of the plot you see:

$$
f(x)
$$

This is the model’s final output for that one person and that one class.

So in the high-probability case:

$$
f(x) = 1
$$

and in the low-probability case:

$$
f(x) \approx 0
$$

That means the model predicted about **100%** probability of `>50K` for the first person, and about **0%** probability for the second person.

---

#### 3. The horizontal scale

The x-axis is the **model output scale for the class being explained**.

```python
model_output="probability"

# Force plots

It shows the same additive idea as the waterfall plot:

- start from a **baseline prediction**
- add feature contributions
- arrive at the **final model prediction**

In SHAP's terminology, the force plot is an **additive force layout**.

The waterfall is slightly easier to follow from personal opinion

In [ ]:
# Initialize SHAP JS support once in the notebook
shap.plots.initjs()

high_force_plot = shap.plots.force(
    adult_shap_explanation_high_class1,
    contribution_threshold=0.08
)

# Save the interactive force plot as HTML
shap.save_html(
    str(FIG_DIR / "session2_03_shap_force_high_case_class1.html"),
    high_force_plot
)


high_force_plot

# note, if you use dark background like me, then this will look strange, the html is saved so you can inspect

In [ ]:
# Interactive HTML force plot for the low-probability case

low_force_plot = shap.plots.force(
    adult_shap_explanation_low_class1,
    contribution_threshold=0.08
)

# Save the interactive force plot as HTML
shap.save_html(
    str(FIG_DIR / "session2_04_shap_force_low_case_class1.html"),
    low_force_plot
)

low_force_plot

## Local explanation method 2: LIME

LIME is another **local explanation method**.

Instead of using Shapley-style feature contributions, LIME explains one prediction by fitting a **simple interpretable model** around the selected observation.

The idea is:

1. choose one observation to explain
2. create many perturbed versions of that observation
3. ask the black-box model for predictions on those perturbed samples
4. weight the perturbed samples by how close they are to the original observation
5. fit a simple local surrogate model
6. interpret that local model as the explanation

So LIME does not directly explain the full Random Forest itself.
It explains the Random Forest **locally around one chosen case** using a simpler surrogate model. In this case the local model is a **weighted ridge regression(linear)**. This can be adapted so you can have linear model, Lasso, since the goal is that it should fit locally around one instance.

In [50]:
adult_numeric_features = X_adult_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

adult_categorical_features = X_adult_train.select_dtypes(
    include=["object", "string"]
).columns.tolist()

In [ ]:
# PREPARE LIME INPUT METADATA
# -----------------------------
# LIME works with the raw tabular input representation.
# So here we use the original training dataframe columns, not the one-hot encoded feature space.

adult_feature_names = X_adult_train.columns.tolist()

# Identify which raw columns are categorical
adult_categorical_feature_indices = [
    i for i, col in enumerate(adult_feature_names)
    if col in adult_categorical_features
]

# readable category-name mappings for categorical columns
# LIME can use these so that categories appear more clearly in the explanation.
adult_categorical_names = {}

for idx, col in enumerate(adult_feature_names):
    if col in adult_categorical_features:
        adult_categorical_names[idx] = sorted(
            X_adult_train[col].astype(str).fillna("missing").unique().tolist()
        )

print("No. of raw input features:", len(adult_feature_names))
print("Categorical feature indices:")
print(adult_categorical_feature_indices)

In [ ]:
# Build LIME-specific encoded copies of the raw data
# --------------------------------------------------
# LIME tabular expects categorical columns to be integer-coded.
# We therefore create encoded versions of the raw input tables just for LIME.

X_adult_train_lime = X_adult_train.copy()
X_adult_example_high_lime = X_adult_example_high.copy()
X_adult_example_low_lime = X_adult_example_low.copy()

adult_categorical_names = {}
adult_lime_category_to_code = {}
adult_lime_code_to_category = {}

for col in adult_categorical_features:
    # Use a consistent placeholder for missing categorical values in the LIME copy
    train_values = X_adult_train_lime[col].astype(object).where(X_adult_train_lime[col].notna(), "missing")
    high_values = X_adult_example_high_lime[col].astype(object).where(X_adult_example_high_lime[col].notna(), "missing")
    low_values = X_adult_example_low_lime[col].astype(object).where(X_adult_example_low_lime[col].notna(), "missing")

    # Collect category levels from the training data
    category_levels = sorted(train_values.unique().tolist())

    # Build mappings
    category_to_code = {category: code for code, category in enumerate(category_levels)}
    code_to_category = {code: category for category, code in category_to_code.items()}

    adult_lime_category_to_code[col] = category_to_code
    adult_lime_code_to_category[col] = code_to_category

    # Save readable names for LIME output
    feature_idx = adult_feature_names.index(col)
    adult_categorical_names[feature_idx] = category_levels

    # Encode the training and example data
    X_adult_train_lime[col] = train_values.map(category_to_code).astype(int)
    X_adult_example_high_lime[col] = high_values.map(category_to_code).astype(int)
    X_adult_example_low_lime[col] = low_values.map(category_to_code).astype(int)

print("LIME-encoded training data shape:", X_adult_train_lime.shape)
print("LIME-encoded high-case shape    :", X_adult_example_high_lime.shape)
print("LIME-encoded low-case shape     :", X_adult_example_low_lime.shape)

display(X_adult_train_lime.head())

This step creates the **LIME explainer object**.

In [56]:
# Build the LIME tabular explainer

# Important:
# LIME expects the training_data passed here to be numeric.
# That is why we created X_adult_train_lime, where the categorical
# columns were converted to integer codes just for LIME.

# training_data:
#   a 2D numeric array that LIME uses to learn the feature distributions
#   and generate perturbed samples around the observation to explain

# mode="classification":
#   tells LIME that the prediction task is classification rather than regression

# feature_names:
#   the original raw input column names, used for readable explanations

# categorical_features:
#   the column indices of the categorical variables in the raw input table
#   LIME needs to know which columns should be treated as categorical

# categorical_names:
#   a mapping from column index to the readable category labels for that column
#   this helps LIME display explanations using category names instead of just codes

# class_names:
#   names of the target classes, in prediction output order

# discretize_continuous=True:
#   asks LIME to turn continuous variables into intervals when forming local rules
#   so explanations may look like "age <= 28" instead of just "age"

# random_state=SEED:
#   keeps LIME's perturbation process reproducible

lime_explainer_adult = LimeTabularExplainer(
    training_data=X_adult_train_lime.to_numpy(),
    mode="classification",
    feature_names=adult_feature_names,
    categorical_features=adult_categorical_feature_indices,
    categorical_names=adult_categorical_names,
    class_names=["<=50K", ">50K"],
    discretize_continuous=True,
    random_state=SEED
)

### prediction wrapper for LIME

LIME perturbs the selected observation and sends the perturbed samples into a prediction function.

In our notebook, those perturbed samples are in the **LIME-encoded tabular format**, where categorical columns have integer codes.

But our fitted Random Forest pipeline was built to receive the **original raw tabular format** with the original category labels.

So this wrapper converts LIME's encoded samples back into the original raw feature representation, then sends them into the fitted pipeline to obtain prediction probabilities.

In [ ]:
# Build a prediction wrapper for LIME
#
# Why is this needed?
# LIME will generate perturbed samples as a NumPy array using the
# LIME-specific encoded representation.
#
# However, our fitted Random Forest pipeline expects a pandas DataFrame
# in the original raw feature format, with the original category labels
# rather than integer codes.
#
# So this wrapper does four things:
# 1. receives LIME's encoded NumPy array
# 2. converts it into a DataFrame with the original raw feature names
# 3. decodes the integer-coded categorical columns back to their original labels
# 4. passes the decoded DataFrame into rf_pipeline_adult.predict_proba(...)

def adult_pipeline_predict_proba_for_lime(lime_input_array):
    # Convert incoming NumPy array to a DataFrame with raw feature names
    lime_input_df = pd.DataFrame(lime_input_array, columns=adult_feature_names)

    # Decode the categorical columns back to their original labels
    for col in adult_categorical_features:
        lime_input_df[col] = (
            lime_input_df[col]
            .round()                    # keep codes as whole numbers
            .astype(int)                # convert to integer codes
            .map(adult_lime_code_to_category[col])  # decode code -> original label
        )

        # If any unseen / invalid codes appear, replace them with "missing"
        lime_input_df[col] = lime_input_df[col].fillna("missing")

    # Return class probabilities from the fitted pipeline
    return rf_pipeline_adult.predict_proba(lime_input_df)

# Quick test on one encoded row
adult_lime_test_probs = adult_pipeline_predict_proba_for_lime(
    X_adult_example_high_lime.to_numpy()
)

print("Wrapper output shape:", adult_lime_test_probs.shape)
print("Wrapper output probabilities:", adult_lime_test_probs)

1 = one input row was passed into the wrapper
2 = two class probabilities were returned

# For the high class probability case

In [ ]:
# Explain the high-probability case with LIME
#
# data_row:
#   one selected observation in the LIME-encoded raw-input format
#
# predict_fn:
#   the wrapper function that converts LIME's encoded rows back to raw labels
#   and then calls the fitted pipeline to get predicted probabilities
#
# num_features=10:
#   show up to 10 local explanation terms
#
# top_labels=1:
#   explain the top predicted class for this observation
lime_explanation_high = lime_explainer_adult.explain_instance(
    data_row=X_adult_example_high_lime.iloc[0].to_numpy(),
    predict_fn=adult_pipeline_predict_proba_for_lime,
    num_features=10,
    top_labels=1
)

print(lime_explanation_high.available_labels())

In [ ]:
# Inspect the high-case LIME explanation as a readable list
#
# as_list(label=1) returns a list of:
#   (human-readable rule, local weight)
#
# Positive weights push the explanation toward the chosen class.
# Negative weights push the explanation away from the chosen class.

adult_class_names = ["<=50K", ">50K"]

# Which class did LIME explain?
adult_lime_high_label = int(lime_explanation_high.available_labels()[0])

# What did the original fitted pipeline predict for this same row?
adult_lime_high_model_probs = adult_pipeline_predict_proba_for_lime(
    X_adult_example_high_lime.to_numpy()
)[0]

print("Explained label:", adult_lime_high_label) 
print("Explained class name:", adult_class_names[adult_lime_high_label])

print("\nModel predicted probabilities for this case:")
print(f"  <=50K : {adult_lime_high_model_probs[0]:.4f}")
print(f"   >50K : {adult_lime_high_model_probs[1]:.4f}")

# The readable LIME explanation:
adult_lime_high_list = lime_explanation_high.as_list(label=adult_lime_high_label)

print("\nLIME local explanation rules and weights:") ###adding some contextual things to help with reading the output
for rule_text, weight in adult_lime_high_list:
    direction = "supports" if weight > 0 else "opposes"
    print(f"{rule_text} -> {weight:.4f} ({direction} class {adult_class_names[adult_lime_high_label]})")

A local surrogate weight is the coefficient or contribution size from the simple interpretable model that LIME fits near one observation, not a weight from the original Random Forest. Molnar describes LIME as building a new local dataset of perturbed samples, weighting those samples by how close they are to the instance of interest, and then fitting an interpretable model on that locally weighted dataset.

In [ ]:
# Plot the high-case LIME explanation as a bar chart
adult_lime_high_fig = lime_explanation_high.as_pyplot_figure(label=adult_lime_high_label)

plt.tight_layout()
plt.savefig(
    FIG_DIR / "session2_05_lime_high_case_barplot.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

In [ ]:
# Save the high-case LIME explanation as HTML
lime_explanation_high.save_to_file(
    str(FIG_DIR / "session2_06_lime_high_case_explanation.html")
)

In the high-probability case, LIME says the local explanation for >50K is driven mostly by positive capital gain, then married status, higher education, and longer working hours, with zero capital loss acting slightly in the opposite direction.

LIME is a **local surrogate explanation** method.

It does not explain the full Random Forest directly.
Instead, it builds a simple local model around one selected case and uses that local model to explain the prediction.

So the important LIME output is:

- a list of **rules**
- with associated **local weights**

Each line in the explanation has the form:

`rule : weight`

For example:

`capital_gain > 0.00 : 0.4532`


#### Interpret the weights

- **positive weight** = supports the explained class
- **negative weight** = argues against the explained class

These weights are **local surrogate weights**, not SHAP values and not direct probability changes.
